<img src="https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/_assets/logo_sigmoidal.png" alt="Sigmoidal" width="400">

# Detecção de Blur com FFT em Imagens Reais

Notebook público do artigo publicado no blog [Sigmoidal](https://sigmoidal.ai).

---

In [ ]:
import urllib.request
import numpy as np
import cv2
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'image.cmap': 'gray',
})

## Blur como filtragem passa-baixas

O **blur** (desfoque) é o resultado da convolução da imagem original $f(x, y)$ com uma função de espalhamento (*Point Spread Function*) $h(x, y)$:

$$g(x, y) = f(x, y) * h(x, y)$$

No domínio da frequência, a convolução se torna uma **multiplicação**:

$$G(u, v) = F(u, v) \cdot H(u, v)$$

Se $h$ é um kernel gaussiano, então $H(u, v)$ se comporta como um **filtro passa-baixas** — preserva baixas frequências e atenua altas frequências (bordas e detalhes finos).

A consequência prática:

- Imagem **nítida** → muita energia em altas frequências.
- Imagem **borrada** → altas frequências atenuadas → espectro concentrado perto da origem.

## Transformada de Fourier 2D

Para uma imagem em tons de cinza $f(x, y)$ de tamanho $M \times N$, a Transformada Discreta de Fourier 2D é:

$$F(u, v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x, y)\, e^{-j\,2\pi\left(\frac{ux}{M} + \frac{vy}{N}\right)}$$

Na prática, usamos `np.fft.fft2` para calcular a FFT 2D e `np.fft.fftshift` para centralizar o componente DC (baixas frequências) no centro da imagem.

Para visualizar o espectro, aplicamos a magnitude em escala logarítmica:

$$\text{magnitude}(u, v) = \log_{10}\!\big(1 + |F(u, v)|\big)$$

## Carregando as imagens

In [ ]:
# URLs das imagens hospedadas no GitHub
URL_NITIDA = "https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/Deteccao-de-Blur-com-FFT/imagem_nitida.jpg"
URL_BORRADA = "https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/Deteccao-de-Blur-com-FFT/imagem_borrada.jpg"


def url_to_image(url):
    """Faz download de uma imagem a partir da URL e retorna como array BGR."""
    resp = urllib.request.urlopen(url)
    data = np.asarray(bytearray(resp.read()), dtype="uint8")
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def resize_width(img, width=500):
    """Redimensiona mantendo a proporção, fixando a largura."""
    h, w = img.shape[:2]
    escala = width / float(w)
    return cv2.resize(img, (width, int(h * escala)), interpolation=cv2.INTER_AREA)


# Carrega, redimensiona e converte para tons de cinza
img_nitida = resize_width(url_to_image(URL_NITIDA), width=500)
img_borrada = resize_width(url_to_image(URL_BORRADA), width=500)

gray_nitida = cv2.cvtColor(img_nitida, cv2.COLOR_BGR2GRAY)
gray_borrada = cv2.cvtColor(img_borrada, cv2.COLOR_BGR2GRAY)

# Exibir lado a lado
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(cv2.cvtColor(img_nitida, cv2.COLOR_BGR2RGB))
axes[0].set_title('Imagem nítida')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_borrada, cv2.COLOR_BGR2RGB))
axes[1].set_title('Imagem borrada')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Visualizando o espectro de frequência

Antes de construir o detector, vamos visualizar os espectros de magnitude das duas imagens para confirmar que a imagem borrada tem menos energia nas altas frequências.

In [ ]:
def plot_espectro(image, titulo):
    """Calcula e exibe a imagem lado a lado com seu espectro de magnitude."""
    fft = np.fft.fft2(image)
    fft_shift = np.fft.fftshift(fft)
    magnitude = np.log10(1 + np.abs(fft_shift))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title(titulo)
    axes[0].axis('off')

    im = axes[1].imshow(magnitude, cmap='inferno')
    axes[1].set_title('Espectro de magnitude')
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


plot_espectro(gray_nitida, 'Imagem nítida')
plot_espectro(gray_borrada, 'Imagem borrada')

## Implementação do detector de blur via FFT

A ideia é simples:

1. Calcular a FFT 2D e centralizar com `fftshift`.
2. Criar uma **máscara circular** de raio `size` e zerar essa região central — removendo as baixas frequências.
3. Aplicar a FFT inversa para reconstruir apenas o conteúdo de **altas frequências**.
4. Calcular a **média da magnitude** desse sinal — quanto maior, mais detalhes finos (mais nítida).
5. Comparar com um limiar `thresh`: se a média estiver abaixo, a imagem é considerada borrada.

Essa abordagem é uma versão didática inspirada em [Liu et al. (CVPR 2008)](https://ieeexplore.ieee.org/document/4587825), que propôs detectar blur parcial em imagens analisando o conteúdo espectral por patches.

In [ ]:
def detect_blur_fft(image, size=60, thresh=10):
    """
    Detecta se uma imagem em tons de cinza esta borrada usando FFT.

    Parametros:
        image  : imagem em tons de cinza (numpy array 2D)
        size   : raio da mascara circular central (baixas frequencias)
        thresh : limiar de nitidez (calibrar conforme a aplicacao)

    Retorna:
        (mean_mag, is_blurry)
        mean_mag  : media da magnitude das altas frequencias (>= 0)
        is_blurry : True se a imagem for considerada borrada
    """
    if image.ndim != 2:
        raise ValueError("A imagem deve ser 2D (tons de cinza).")

    h, w = image.shape
    cy, cx = h // 2, w // 2

    # FFT 2D + centralizar componente DC
    fft = np.fft.fft2(image)
    fft_shift = np.fft.fftshift(fft)

    # Mascara circular: zerar baixas frequencias (raio <= size)
    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
    fft_shift[dist <= size] = 0

    # FFT inversa: reconstruir apenas altas frequencias
    recon = np.fft.ifft2(np.fft.ifftshift(fft_shift))

    # Metrica: media da magnitude (sempre >= 0)
    mean_mag = np.mean(np.abs(recon))
    is_blurry = mean_mag <= thresh

    return mean_mag, is_blurry

## Aplicando o detector

In [ ]:
# Hiperparametros
SIZE = 60     # raio da mascara circular
THRESH = 10   # limiar de nitidez

mean_nitida, blur_nitida = detect_blur_fft(gray_nitida, size=SIZE, thresh=THRESH)
mean_borrada, blur_borrada = detect_blur_fft(gray_borrada, size=SIZE, thresh=THRESH)

print(f"Imagem nitida  — metrica FFT: {mean_nitida:.2f} | {'BORRADA' if blur_nitida else 'NITIDA'}")
print(f"Imagem borrada — metrica FFT: {mean_borrada:.2f} | {'BORRADA' if blur_borrada else 'NITIDA'}")

In [ ]:
# Comparacao visual com anotacao
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, img, mean_val, is_blurry, titulo in [
    (axes[0], gray_nitida, mean_nitida, blur_nitida, 'Imagem nítida'),
    (axes[1], gray_borrada, mean_borrada, blur_borrada, 'Imagem borrada'),
]:
    ax.imshow(img, cmap='gray')
    rotulo = 'BORRADA' if is_blurry else 'NÍTIDA'
    cor = '#e74c3c' if is_blurry else '#2ecc71'
    ax.set_title(f'{titulo}\n{rotulo} (métrica = {mean_val:.2f})', color=cor, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Experimento: blur progressivo

Para validar o detector, vamos aplicar borramentos gaussianos com kernels crescentes na imagem nítida e observar como a métrica decai.

In [ ]:
kernels = list(range(1, 32, 2))  # valores impares: 1, 3, 5, ..., 31
scores_fft = []

for k in kernels:
    img_blur = cv2.GaussianBlur(gray_nitida, (k, k), 0) if k > 1 else gray_nitida.copy()
    mean_val, is_blurry = detect_blur_fft(img_blur, size=SIZE, thresh=THRESH)
    scores_fft.append(mean_val)
    status = 'BORRADA' if is_blurry else 'NITIDA'
    print(f"Kernel {k:2d} — metrica: {mean_val:8.2f} — {status}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(kernels, scores_fft, 'o-', color='#2c3e50', linewidth=2, markersize=6)
plt.axhline(y=THRESH, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Limiar = {THRESH}')
plt.fill_between(kernels, 0, THRESH, alpha=0.08, color='red')
plt.xlabel('Tamanho do kernel gaussiano')
plt.ylabel('Métrica FFT (média da magnitude)')
plt.title('Degradação da métrica FFT sob blur progressivo')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Comparação: FFT vs. Variância do Laplaciano

O método mais simples e popular para detecção de blur é a **variância do Laplaciano** ([Pech-Pacheco et al., ICPR 2000](https://ieeexplore.ieee.org/document/903548)): aplicar o operador Laplaciano (segunda derivada) e calcular a variância do resultado.

- Imagem nítida → Laplaciano forte → variância alta.
- Imagem borrada → Laplaciano fraco → variância baixa.

Vamos comparar as duas abordagens no mesmo experimento de blur progressivo.

In [ ]:
def variancia_laplaciano(image):
    """Calcula a variancia do Laplaciano como metrica de nitidez."""
    return cv2.Laplacian(image, cv2.CV_64F).var()


# Calcular metrica do Laplaciano para cada nivel de blur
scores_lap = []
for k in kernels:
    img_blur = cv2.GaussianBlur(gray_nitida, (k, k), 0) if k > 1 else gray_nitida.copy()
    scores_lap.append(variancia_laplaciano(img_blur))

# Normalizar ambas as metricas para [0, 1] (para comparacao visual)
fft_norm = np.array(scores_fft) / max(scores_fft)
lap_norm = np.array(scores_lap) / max(scores_lap)

plt.figure(figsize=(10, 5))
plt.plot(kernels, fft_norm, 'o-', label='FFT (média da magnitude)', linewidth=2, markersize=6)
plt.plot(kernels, lap_norm, 's--', label='Laplaciano (variância)', linewidth=2, markersize=6)
plt.xlabel('Tamanho do kernel gaussiano')
plt.ylabel('Métrica normalizada [0, 1]')
plt.title('FFT vs. Laplaciano — resposta ao blur progressivo')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Visualizando o efeito do blur no espectro

Para consolidar a intuição, vamos comparar o espectro da imagem original com o de versões progressivamente borradas.

In [ ]:
niveis_blur = [1, 5, 11, 21, 31]

fig, axes = plt.subplots(2, len(niveis_blur), figsize=(16, 7))

for i, k in enumerate(niveis_blur):
    img_blur = cv2.GaussianBlur(gray_nitida, (k, k), 0) if k > 1 else gray_nitida.copy()

    # Imagem
    axes[0, i].imshow(img_blur, cmap='gray')
    axes[0, i].set_title(f'Kernel = {k}', fontsize=11)
    axes[0, i].axis('off')

    # Espectro
    fft_shift = np.fft.fftshift(np.fft.fft2(img_blur))
    mag = np.log10(1 + np.abs(fft_shift))
    axes[1, i].imshow(mag, cmap='inferno')
    axes[1, i].set_title('Espectro', fontsize=11)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Imagem', fontsize=12)
axes[1, 0].set_ylabel('Espectro FFT', fontsize=12)
plt.suptitle('Efeito do blur gaussiano no espectro de frequência', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Limitações do método

Alguns cenários em que o detector FFT pode falhar:

- **Imagens com pouca textura** (céu limpo, parede lisa): poucas altas frequências mesmo sem blur, gerando falsos positivos.
- **Blur parcial**: se apenas parte da imagem está borrada (foco seletivo), a métrica global pode não capturar.
- **Blur de movimento vs. gaussiano**: o detector trata ambos de forma igual, mas a assinatura espectral é diferente (blur de movimento atenua frequências em uma direção específica).
- **Sensibilidade ao limiar**: o valor de `thresh` depende da resolução, do conteúdo e do tipo de imagem — precisa ser calibrado para cada aplicação.

Para cenários mais robustos, considere:
- Análise **por patches** (como no paper original de Liu et al.);
- Métricas complementares (Laplaciano, gradientes);
- Modelos de deep learning treinados para classificação de blur.

## Principais conclusões

- O **blur gaussiano** atua como filtro **passa-baixas** no domínio da frequência, atenuando bordas e detalhes finos.
- A **FFT 2D** permite decompor a imagem em componentes de frequência e medir diretamente o conteúdo de altas frequências.
- O detector funciona zerando a região central do espectro (baixas frequências) e medindo a **magnitude restante** — quanto menor, mais borrada a imagem.
- A **variância do Laplaciano** é uma alternativa simples e eficaz que costuma ter desempenho comparável.
- Ambos os métodos servem como baseline sólido para controle de qualidade de imagens em pipelines de visão computacional.